# Schema Pruning Demo (BIRD) - V2 (Embedding-Only, No BM25)
Notebook ini menggunakan EmbeddingRetrieverV2 dalam mode embedding murni (lexical_weight=0.0) dan tanpa mandatory key preservation.


In [9]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists():
    for parent in PROJECT_ROOT.parents:
        if (parent / "src").exists() and (parent / "Dataset").exists():
            PROJECT_ROOT = parent
            break

SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from pruning.schema_loader import SchemaLoader
from pruning.embedding_retriever_v2 import EmbeddingRetrieverV2
from pruning.pruner_v2 import SchemaPrunerV2
from pruning.evaluation import extract_gold_columns, precision_recall_f1_f6
from bird_datasets.bird_dataset_loader import BirdDatasetLoader

DATA_ROOT = PROJECT_ROOT / 'Dataset' / 'BIRD' / 'dev'
DATASET_PATH = DATA_ROOT / 'dev.json'
TABLES_PATH = DATA_ROOT / 'dev_tables.json'
DB_ROOT = DATA_ROOT / 'dev_databases' / 'dev_databases'

print('Project root:', PROJECT_ROOT)
print('Tables path:', TABLES_PATH)
print('DB root:', DB_ROOT)

Project root: d:\Kuliah\Skripsi\Schema Pruning
Tables path: d:\Kuliah\Skripsi\Schema Pruning\Dataset\BIRD\dev\dev_tables.json
DB root: d:\Kuliah\Skripsi\Schema Pruning\Dataset\BIRD\dev\dev_databases\dev_databases


In [ ]:
# Input manual user (edit langsung variabel di bawah ini)
db_id = 'california_schools'
question = 'What is the highest eligible free rate for K-12 students in the schools in Alameda County?'
gold_sql = (
    "SELECT `Free Meal Count (K-12)` / `Enrollment (K-12)` FROM frpm WHERE `County Name` = 'Alameda' ORDER BY (CAST(`Free Meal Count (K-12)` AS REAL) / `Enrollment (K-12)`) DESC LIMIT 1"
)
evidence = ''
top_k = 40
lexical_weight = 0.0  # fixed: embedding-only (BM25 disabled)
preserve_mandatory_keys = False

In [11]:
# Pilih sample dari dev.json (opsional)
use_sample_id = False
sample_id = 0  # ganti sesuai kebutuhan
if use_sample_id:
    import json
    samples = json.load(open(DATASET_PATH, 'r', encoding='utf-8'))
    item = samples[sample_id]
    db_id = item['db_id']
    question = item['question']
    gold_sql = item.get('SQL', item.get('sql', ''))
    evidence = item.get('evidence', '')
    print('Loaded sample_id:', sample_id)
    print('db_id:', db_id)
    print('question:', question)

In [12]:
# Alternatif input interaktif (set True jika ingin pakai input())
use_interactive = False
if use_interactive:
    db_id = input('db_id: ').strip()
    question = input('question: ').strip()
    gold_sql = input('gold SQL (boleh kosong): ').strip()
    if gold_sql == '':
        gold_sql = ''
    evidence = input('evidence (boleh kosong): ').strip()
    top_k = int(input('top_k (int): ').strip())
    lexical_weight = 0.0  # locked to embedding-only mode
    preserve_mandatory_keys = input('preserve_mandatory_keys (True/False): ').strip().lower() in ('1', 'true', 'yes', 'y')

In [13]:
schema_loader = SchemaLoader(TABLES_PATH, DB_ROOT)
if lexical_weight != 0.0:
    raise ValueError('Notebook ini khusus embedding-only: set lexical_weight = 0.0')

retriever = EmbeddingRetrieverV2(
    lexical_weight=lexical_weight,
    use_evidence=True,
)
pruner = SchemaPrunerV2(schema_loader, retriever)

schema = schema_loader.get_schema(db_id)
pruning = pruner.prune(
    question,
    db_id,
    top_k,
    evidence=evidence,
    preserve_mandatory_keys=preserve_mandatory_keys,
)
gold_columns = extract_gold_columns(gold_sql or '', schema)
precision, recall, f1, f6 = precision_recall_f1_f6(pruning.final_columns, gold_columns)

print('Retriever backend:', pruning.backend)
print('Lexical weight (fixed):', lexical_weight)
print('Preserve mandatory keys:', preserve_mandatory_keys)
print('Top-k columns:', pruning.top_k_columns)
print('Preserved columns:', pruning.preserved_columns)
print('Final columns:', pruning.final_columns)
print('Gold columns:', sorted(gold_columns))
print(f'Precision: {precision:.4f} | Recall: {recall:.4f} | F1: {f1:.4f} | F6: {f6:.4f}')

Retriever backend: embedding
Lexical weight (fixed): 0.0
Preserve mandatory keys: False
Top-k columns: ['frpm.Percent (%) Eligible Free (K-12)', 'frpm.Percent (%) Eligible Free (Ages 5-17)', 'frpm.Enrollment (Ages 5-17)', 'frpm.Enrollment (K-12)', 'frpm.Free Meal Count (Ages 5-17)', 'frpm.Free Meal Count (K-12)', 'schools.District', 'schools.County', 'frpm.Percent (%) Eligible FRPM (Ages 5-17)', 'schools.NCESSchool', 'frpm.Percent (%) Eligible FRPM (K-12)', 'schools.NCESDist', 'schools.School', 'schools.State', 'schools.Charter', 'frpm.2013-14 CALPADS Fall 1 Certification Status', 'schools.Zip', 'schools.OpenDate', 'frpm.Charter School (Y/N)', 'schools.ClosedDate', 'schools.CharterNum', 'schools.MailState', 'schools.GSoffered', 'schools.StreetAbr', 'schools.FundingType', 'schools.City', 'schools.StatusType', 'schools.Street', 'frpm.FRPM Count (K-12)', 'schools.MailCity']
Preserved columns: []
Final columns: ['frpm.2013-14 CALPADS Fall 1 Certification Status', 'frpm.Charter School (Y/N)

## Pengujian semua dataset
Cell ini mengevaluasi seluruh sample di DATASET_PATH (default: dev.json).
Mode: embedding-only (BM25 nonaktif), preserve_mandatory_keys=False.


In [14]:
from statistics import mean
from itertools import islice

dataset = BirdDatasetLoader(DATASET_PATH)
max_samples = None  # ganti int untuk uji cepat

precisions, recalls, f1s, f6s, reductions = [], [], [], [], []
full_recall_flags = []
per_sample = []

for idx, sample in enumerate(dataset.iter_samples()):
    if max_samples is not None and idx >= max_samples:
        break

    schema = schema_loader.get_schema(sample.db_id)
    pruning = pruner.prune(
        sample.question,
        sample.db_id,
        top_k,
        evidence=sample.evidence,
        preserve_mandatory_keys=preserve_mandatory_keys,
    )
    gold_columns = extract_gold_columns(sample.sql, schema)

    precision, recall, f1, f6 = precision_recall_f1_f6(
        pruning.final_columns, gold_columns
    )

    total_cols = len(schema.columns)
    pruned_cols = len(pruning.final_columns)
    reduction = 1.0 - (pruned_cols / total_cols) if total_cols else 0.0

    is_full_recall = recall >= 0.99999

    per_sample.append({
        "question_id": sample.question_id,
        "db_id": sample.db_id,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "f6": f6,
        "reduction_ratio": reduction,
        "total_1_recall": is_full_recall,
    })

    precisions.append(precision)
    recalls.append(recall)
    f1s.append(f1)
    f6s.append(f6)
    reductions.append(reduction)
    full_recall_flags.append(is_full_recall)

if per_sample:
    total_full_recall = sum(full_recall_flags)

    print("Retriever backend:", pruner.retriever.backend)
    print("Lexical weight:", lexical_weight)
    print("Preserve mandatory keys:", preserve_mandatory_keys)
    print("Samples:", len(per_sample))
    print(f"Mean Precision: {mean(precisions):.4f}")
    print(f"Mean Recall: {mean(recalls):.4f}")
    print(f"Mean F1: {mean(f1s):.4f}")
    print(f"Mean F6: {mean(f6s):.4f}")
    print(f"Mean Reduction Ratio: {mean(reductions):.4f}")
    print(f"Total samples with 100% recall: {total_full_recall} / {len(per_sample)}")

    method_name = "embedding_only_mkp" if preserve_mandatory_keys else "embedding_only"
    aggregate = {
        "num_samples": len(per_sample),
        "top_k": top_k,
        "mean_precision": mean(precisions),
        "mean_recall": mean(recalls),
        "mean_f1": mean(f1s),
        "mean_f6": mean(f6s),
        "mean_reduction_ratio": mean(reductions),
        "total_full_recall": total_full_recall,
        "method": method_name,
        "retriever_backend": pruner.retriever.backend,
        "lexical_weight": lexical_weight,
        "preserve_mandatory_keys": preserve_mandatory_keys,
    }

    from utils.results_writer import write_metrics_txt
    metrics_dir = PROJECT_ROOT / "outputs" / "pruning stats"
    txt_path = metrics_dir / f"aggregate_v2_{method_name}_top_{top_k}.txt"
    write_metrics_txt(
        txt_path,
        aggregate,
        title=f"Benchmark Summary V2 ({method_name}, top_k={top_k})",
    )
    print("Saved:", txt_path)
else:
    print("No samples evaluated.")

Retriever backend: embedding
Lexical weight: 0.0
Preserve mandatory keys: False
Samples: 1534
Mean Precision: 0.1370
Mean Recall: 0.8174
Mean F1: 0.2249
Mean F6: 0.6718
Mean Reduction Ratio: 0.4784
Total samples with 100% recall: 914 / 1534
Saved: d:\Kuliah\Skripsi\Schema Pruning\outputs\pruning stats\aggregate_v2_embedding_only_top_30.txt


## Ekspor dataset JSON untuk LLM
Cell ini membuat dataset JSON/JSONL berisi pertanyaan + schema prompt (embedding-only, BM25 nonaktif, tanpa mandatory key preservation) untuk inference LLM.
Disertakan gold_sql dan gold_columns untuk evaluasi/analisis.


In [ ]:
# # Ekspor dataset untuk LLM (embedding-only)
# import json
# from pathlib import Path

# dataset = BirdDatasetLoader(DATASET_PATH)
# max_samples = None  # ganti int untuk uji cepat
# use_evidence_prompt = True

# SYSTEM_PROMPT = "You are an expert text-to-SQL assistant for SQLite. Use only the provided schema and question. Output only the final SQL query as plain text. Do not add explanations or formatting. Quote identifiers with backticks if they contain spaces or symbols."

# def quote_column_full_name(full_name):
#     table, col = full_name.split(".", 1)
#     return f"{table}.`{col}`"

# def build_schema_prompt(schema, final_columns):
#     final_set = set(final_columns)
#     pk_set = {c.full_name for c in schema.primary_keys}
#     table_to_cols = {}
#     # Preserve schema order by iterating schema.columns
#     for col in schema.columns:
#         if col.full_name in final_set:
#             table_to_cols.setdefault(col.table, []).append(col)

#     def q(name):
#         return f"`{name}`"

#     def col_type(col):
#         return col.col_type if col.col_type else "text"

#     fk_by_table = {}
#     for col_a, col_b in schema.foreign_keys:
#         if col_a.full_name in final_set and col_b.full_name in final_set:
#             fk_by_table.setdefault(col_a.table, []).append((col_a, col_b))

#     lines = []
#     for table in schema.tables:
#         if table not in table_to_cols:
#             continue
#         cols = table_to_cols[table]
#         pk_cols = [c for c in cols if c.full_name in pk_set]

#         lines.append(f"CREATE TABLE {q(table)} (")
#         col_lines = [f"  {q(c.name)} {col_type(c)}" for c in cols]

#         if pk_cols:
#             pk_list = ", ".join(q(c.name) for c in pk_cols)
#             col_lines.append(f"  PRIMARY KEY ({pk_list})")

#         for col_a, col_b in fk_by_table.get(table, []):
#             col_lines.append(
#                 f"  FOREIGN KEY ({q(col_a.name)}) REFERENCES {q(col_b.table)} ({q(col_b.name)})"
#             )

#         lines.append(",\n".join(col_lines))
#         lines.append(");")
#         lines.append("")

#     return "\n".join(lines).rstrip()
# rows = []
# for idx, sample in enumerate(dataset.iter_samples()):
#     if max_samples is not None and idx >= max_samples:
#         break
#     schema = schema_loader.get_schema(sample.db_id)
#     pruning = pruner.prune(
#         sample.question,
#         sample.db_id,
#         top_k,
#         evidence=sample.evidence,
#         preserve_mandatory_keys=preserve_mandatory_keys,
#     )
#     schema_prompt = build_schema_prompt(schema, pruning.final_columns)
#     gold_columns = extract_gold_columns(sample.sql, schema)
#     gold_columns_raw = sorted(gold_columns)
#     gold_columns_quoted = [quote_column_full_name(c) for c in gold_columns_raw]

#     user_prompt = f"Database: {sample.db_id}\\n\\nSchema:\\n{schema_prompt}\\n\\n"
#     if use_evidence_prompt and sample.evidence:
#         user_prompt += f"Evidence:\\n{sample.evidence}\\n\\n"
#     user_prompt += f"Question: {sample.question}"
#     messages = [
#         {"role": "system", "content": SYSTEM_PROMPT},
#         {"role": "user", "content": user_prompt},
#     ]

#     rows.append({
#         "question_id": sample.question_id,
#         "db_id": sample.db_id,
#         "question": sample.question,
#         "evidence": sample.evidence,
#         "schema_prompt": schema_prompt,
#         "top_k": top_k,
#         "lexical_weight": lexical_weight,
#         "retriever_backend": pruner.retriever.backend,
#         "preserve_mandatory_keys": preserve_mandatory_keys,
#         "gold_sql": sample.sql,
#         "gold_columns": gold_columns_quoted,
#         "gold_columns_raw": gold_columns_raw,
#         "messages": messages,
#     })

# output_dir = PROJECT_ROOT / "outputs" / "dataset" / "embedding_only"
# output_dir.mkdir(parents=True, exist_ok=True)
# json_path = output_dir / f"llm_dataset_v2_embedding_only_top_{top_k}.json"
# jsonl_path = output_dir / f"llm_dataset_v2_embedding_only_top_{top_k}.jsonl"
    
# with json_path.open("w", encoding="utf-8") as f:
#     json.dump(rows, f, ensure_ascii=False, indent=2)

# with jsonl_path.open("w", encoding="utf-8") as f:
#     for row in rows:
#         f.write(json.dumps(row, ensure_ascii=False) + "\\n")

# print("Saved:", json_path)
# print("Saved:", jsonl_path)
# print("Rows:", len(rows))

Saved: d:\Kuliah\Skripsi\Schema Pruning\outputs\dataset\embedding_only\llm_dataset_v2_embedding_only_top_30.json
Saved: d:\Kuliah\Skripsi\Schema Pruning\outputs\dataset\embedding_only\llm_dataset_v2_embedding_only_top_30.jsonl
Rows: 1534


## Benchmark kecil (opsional)
Set run_benchmark = True untuk menjalankan benchmark kecil.

In [ ]:
# from experiments.benchmark_runner_v2 import BenchmarkConfigV2, run_benchmark_v2

# run_benchmark = False
# if run_benchmark:
#     config = BenchmarkConfigV2(
#         dataset_path=DATASET_PATH,
#         tables_path=TABLES_PATH,
#         db_root=DB_ROOT,
#         top_k=top_k,
#         max_samples=50,
#         lexical_weight=lexical_weight,
#         use_evidence=True,
#         preserve_mandatory_keys=preserve_mandatory_keys,
#         method_name='embedding_only' if not preserve_mandatory_keys else 'embedding_only_mkp',
#     )
#     _, aggregate = run_benchmark_v2(config)
#     print('Benchmark summary (V2):')
#     print(aggregate)